Data obtained here: https://zenodo.org/records/14767363
egrid details here: https://www.epa.gov/system/files/documents/2025-01/egrid2023_technical_guide.pdf
ejscreen in action here: https://pedp-ejscreen.azurewebsites.net/
ejscreen documentation here: https://www.epa.gov/system/files/documents/2024-07/ejscreen-tech-doc-version-2-3.pdf
other resource i couldnt figure out: https://dataverse.harvard.edu/dataset.xhtml?persistentId=doi:10.7910/DVN/RLR5AX
ejscreen tool archive: https://screening-tools.com/epa-ejscreen


In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import seaborn as sns
import pymc as pm
import bambi as bmb
import arviz as az
import statsmodels.api as sm
from sklearn.preprocessing import OneHotEncoder


In [1]:
import pandas as pd


use_cols = [
    'ID', 
    'PEOPCOLOR', 'ACSTOTPOP',
    'LOWINCOME', 'ACSIPOVBAS',
    'UNEMPLOYED', 'ACSUNEMPBAS',
    'LINGISO', 'ACSTOTHH',
    'LESSHS', 'ACSEDUCBAS',
    'UNDER5', 'OVER64'
]

# 2. Initialize an empty list to store our mini-results
chunk_results = []

#Processing files in chunks because it was crashing kernel
chunk_size = 5000
file_path = 'data/EJSCREEN_2023_BG_with_AS_CNMI_GU_VI.csv'

# need latin1 because the file has special characters
with pd.read_csv(file_path, chunksize=chunk_size, usecols=use_cols, encoding='latin1', low_memory=False) as reader:
    for chunk in reader:
        # Create County FIPS on the fly
        chunk['County_FIPS'] = chunk['ID'].astype(str).str.zfill(12).str[:5]
        
        # Aggregate JUST this chunk by County FIPS
        # (It's okay if a county is split across chunks; we will sum them again at the end)
        agg_chunk = chunk.groupby('County_FIPS')[use_cols[1:]].sum()
        
        # Store this small aggregated piece
        chunk_results.append(agg_chunk)



In [3]:
# 4. Concatenate all the small pieces and sum them one last time
# This combines the data from counties that were split across different chunks
final_df = pd.concat(chunk_results).groupby(level=0).sum().reset_index()

# 5. Now calculate your percentages on the final, small dataframe
final_df['Pct_People_of_Color'] = (final_df['PEOPCOLOR'] / final_df['ACSTOTPOP']) * 100
final_df['Pct_Low_Income'] = (final_df['LOWINCOME'] / final_df['ACSIPOVBAS']) * 100
final_df['Pct_Unemployment'] = (final_df['UNEMPLOYED'] / final_df['ACSUNEMPBAS']) * 100
final_df['Pct_Limited_English'] = (final_df['LINGISO'] / final_df['ACSTOTHH']) * 100
final_df['Pct_Less_Than_HS'] = (final_df['LESSHS'] / final_df['ACSEDUCBAS']) * 100
final_df['Pct_Under_5'] = (final_df['UNDER5'] / final_df['ACSTOTPOP']) * 100
final_df['Pct_Over_64'] = (final_df['OVER64'] / final_df['ACSTOTPOP']) * 100

final_df

,County_FIPS,PEOPCOLOR,ACSTOTPOP,LOWINCOME,ACSIPOVBAS,UNEMPLOYED,ACSUNEMPBAS,LINGISO,ACSTOTHH,LESSHS,ACSEDUCBAS,UNDER5,OVER64,Pct_People_of_Color,Pct_Low_Income,Pct_Unemployment,Pct_Limited_English,Pct_Less_Than_HS,Pct_Under_5,Pct_Over_64
0,00000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,00780,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,01001,15668.0,58239.0,17782.0,57790.0,752.0,26623.0,32.0,21856.0,4126.0,39614.0,3318.0,8815.0,26.902934,30.770029,2.824625,0.146413,10.415510,5.697213,15.135905
3,01003,39583.0,227131.0,57840.0,223772.0,3994.0,108361.0,730.0,87190.0,14555.0,161977.0,12035.0,46805.0,17.427388,25.847738,3.685828,0.837252,8.985844,5.298704,20.607051
4,01005,13991.0,25259.0,11195.0,22250.0,808.0,9369.0,117.0,9088.0,4378.0,17995.0,1320.0,4801.0,55.390158,50.314607,8.624186,1.287412,24.328980,5.225860,19.007087
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3218,72145,53453.0,54544.0,40080.0,54242.0,3482.0,19789.0,13350.0,19799.0,9466.0,39632.0,2192.0,11463.0,97.999780,73.891081,17.595634,67.427648,23.884740,4.018774,21.016060
3219,72147,7808.0,8317.0,7186.0,8317.0,358.0,2355.0,1782.0,2374.0,1613.0,5970.0,401.0,1904.0,93.880005,86.401347,15.201699,75.063184,27.018425,4.821450,22.892870
3220,72149,22289.0,22341.0,17790.0,22207.0,1464.0,7856.0,5780.0,7823.0,3329.0,15523.0,1002.0,4188.0,99.767244,80.109875,18.635438,73.884699,21.445597,4.485028,18.745804
3221,72151,31020.0,31047.0,24486.0,31042.0,1506.0,9897.0,8645.0,11905.0,6097.0,22690.0,1092.0,6801.0,99.913035,78.880227,15.216732,72.616548,26.870868,3.517248,21.905498


In [ ]:
final_df.to_csv('data/EJScreen_DEMO23.csv')